# LLM APIs, keys, bootstrap code

## Gemini API
Intro notebook, API calls, Keys, micro projects

In [55]:
!pip install -q -U google-genai python-dotenv

In [59]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads API key from .env file

True

In [60]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [62]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words",
)
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='It learns from data to make smart decisions.'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='nTiaadPWK8Gtz7IPlpif6QU',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=9,
    prompt_token_count=9,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=9
      ),
    ],
    thoughts_token_count=940,
    total_token_count=958
  )
)

In [63]:
print(response.text)
print(f"Billing Info:")
print(f"prompt_token_count={response.usage_metadata.prompt_token_count}")
print(f"candidates_token_count={response.usage_metadata.candidates_token_count}")
print(f"thoughts_token_count={response.usage_metadata.thoughts_token_count}")
print(f"total_token_count={response.usage_metadata.total_token_count}")
print(f"Model Used: {response.model_version}")

It learns from data to make smart decisions.
Billing Info:
prompt_token_count=9
candidates_token_count=9
thoughts_token_count=940
total_token_count=958
Model Used: gemini-2.5-flash


## 1. AI Content Writer

In [37]:
topic = "India"
instructions = "As a expert in the field, write a 100 word essay about - "
prompt = instructions + topic

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=prompt
)
print(response.text)

os.makedirs("output", exist_ok=True)
fileName = "output/" + topic.lower() + ".txt"
with open(fileName, "w") as f:
    f.write(response.text)

India stands as a testament to the coexistence of ancient heritage and modern ambition. As the world’s most populous democracy, it represents a complex mosaic of linguistic, religious, and cultural diversity. Historically a cradle of civilization, contemporary India has transitioned into a global economic powerhouse, leading in information technology, pharmaceuticals, and space exploration. Its strategic location and youthful demographic dividend position it as a critical player in the 21st-century geopolitical landscape. Balancing the challenges of rapid urbanization with sustainable growth, India continues to redefine the paradigm of a pluralistic society moving toward a prosperous, technologically advanced, and inclusive future.


741

## 2. AI Bulk Content Writer
Loop through a list of topics, generate an essay for each, and save them all to separate files.     

In [65]:
import time

topics = [
    "India", 
    "United States", 
    "Peru", 
    "United Kingdom", 
    "South Africa", 
    "Australia",
    "Antarctica"
    ]

os.makedirs("output", exist_ok=True)

for topic in topics:
    instructions = "As a expert in the field, write a 100 word essay about - "
    prompt = instructions + topic

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash", contents=prompt
            )
            break
        except Exception as e:
            print(f"[WARN] {topic}: Attempt {attempt+1} failed, retrying in 10s...")
            time.sleep(10)
    else:
        print(f"[ERROR] {topic}: Skipped after 3 failed attempts")
        continue

    fileName = "output/" + topic.lower() + ".txt"
    with open(fileName, "w") as f:
        f.write(response.text)
    print(f"[INFO] Generated {topic} → {fileName}")
    time.sleep(5)  # pause between requests to avoid rate limits

print("\nDone!")

779

[INFO] Generated India → output/india.txt


694

[INFO] Generated United States → output/united states.txt


724

...

[INFO] Generated South Africa → output/south africa.txt
[WARN] Australia: Attempt 1 failed, retrying in 10s...
[WARN] Australia: Attempt 2 failed, retrying in 10s...


670

[INFO] Generated Australia → output/australia.txt
[WARN] Antarctica: Attempt 1 failed, retrying in 10s...


798

[INFO] Generated Antarctica → output/antarctica.txt

Done!


## 3. AI Chatbot

In [53]:
instructions = "You are a helpful assistant. Answer the user's question based on the conversation history and your knowledge. If you don't know the answer, say you don't know. Answer in less than 50 words.\n"
while True:
    userText = input("User : (write Q to exit) ")
    if userText == 'Q':
        break
    
    question = "\nUser: " + userText
    prompt = instructions + question

    response = client.models.generate_content(
                    model="gemini-2.5-flash", contents=prompt)
    
    print(f"\nUser : {userText}")
    print(f"Gemini : {response.text}")


User : My dogs name is Tony, german shepard, 2 years, tell me more about them
Gemini : Tony, as a 2-year-old German Shepherd, is likely intelligent, loyal, and still very active. At this age, they are typically past the puppy stage, but retain their playful nature. They thrive on mental stimulation and physical exercise, making them highly trainable and dedicated companions.

User : hat is the capital of United States?
Gemini : The capital of the United States is Washington, D.C.

User : Does Washington have snow?
Gemini : Yes, Washington State definitely gets snow. It's common in the mountains (Cascades, Olympics) and eastern Washington. Western lowlands, including Seattle, also experience snow, though typically less frequently and in smaller amounts.

User : Will Tony like snow?
Gemini : I don't have enough information about Tony to know if he will like snow.



### Analysis
The above chatbox looses context of the conversation with each new request. When the user asks if Washington has snow, it tells about Washington State instead of D.C. It did not retain information about Tony, and could not answer anything about it in the next request. 

### Chatbot with conversation history

In [47]:
context = ""
instructions = "You are a helpful assistant. Answer the user's question based on the conversation history and your knowledge. If you don't know the answer, say you don't know. Answer in less than 50 words.\n"
while True:
    userText = input("User : (write Q to exit) ")
    if userText == 'Q':
        break
    
    question = "\nUser: " + userText
    prompt = instructions + context + question

    response = client.models.generate_content(
                    model="gemini-2.5-flash", contents=prompt)
    
    print(f"\nUser : {userText}")
    print(f"Gemini : {response.text}")
    context += f"\nUser: {userText}\nGemini: {response.text}"


User : My dogs name is Tony, german shepard, 2 years, tell me more about them
Gemini : Tony, your 2-year-old German Shepherd, is likely intelligent, loyal, and energetic. This breed is known for its trainability and strong protective instincts, making them excellent companions and often good at various dog sports or work roles. Ensure plenty of exercise and mental stimulation.

User : What is the capital of United States?
Gemini : The capital of the United States is Washington, D.C.

User : Does Washington have snow?
Gemini : Yes, Washington, D.C. does experience snow. It typically receives a few snowfalls each winter, ranging from light dustings to occasional heavier snowstorms.

User : Will Tony like snow?
Gemini : Given Tony is a German Shepherd, a breed known for its thick double coat and active nature, he will likely enjoy playing in the snow. Many GSDs are well-suited to colder weather and find snow stimulating and fun for play.

User : q
Gemini : I'm sorry, I don't understand y

In [50]:
print(context)


User: My dogs name is Tony, german shepard, 2 years, tell me more about them
Gemini: Tony, your 2-year-old German Shepherd, is likely intelligent, loyal, and energetic. This breed is known for its trainability and strong protective instincts, making them excellent companions and often good at various dog sports or work roles. Ensure plenty of exercise and mental stimulation.
User: What is the capital of United States?
Gemini: The capital of the United States is Washington, D.C.
User: Does Washington have snow?
Gemini: Yes, Washington, D.C. does experience snow. It typically receives a few snowfalls each winter, ranging from light dustings to occasional heavier snowstorms.
User: Will Tony like snow?
Gemini: Given Tony is a German Shepherd, a breed known for its thick double coat and active nature, he will likely enjoy playing in the snow. Many GSDs are well-suited to colder weather and find snow stimulating and fun for play.
User: q
Gemini: I'm sorry, I don't understand your last quest

In [52]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='I\'m sorry, I don\'t understand your last question. Could you please clarify what you mean by "q"?'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='PTKaaa7XLs-rmtkPgbKT6Aw',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=25,
    prompt_token_count=267,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=267
      ),
    ],
    thoughts_token_count=122,
    total_token_count=414
  )
)

### Analysis
This chatbot is passed the conversation history along with every new question. It is able to understand the user is talking about Washington DC without giving the context again. It was able to answer questions related to Tony who was introduced in the first statement.

## Overall Analysis

**Context** </br>
LLMs are stateless by default, every prompt needs a good instruction and a context (such as, conversation history, topic of discussion, etc) to get better and _controlled_ responses.

**Token Length** </br>
Better prompts with more instructions and context yield better responses, but at the cost of higher input tokens, which increases billing costs on the API key.

**Rate-Limiter** </br>
Most APIs are bounded by rate limits to control attacks, scale infrastructure, and ensure fair usage across all users. In our bulk writer, we hit a 503 error when sending too many requests in quick succession. Adding retry logic with delays between requests and backoff on failures helps stay within limits and makes batch operations reliable.